In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from category_encoders import MEstimateEncoder
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import StackingRegressor
from catboost import CatBoostRegressor

In [ ]:
df = pd.read_csv('/kaggle/input/competitions/triagegeist/train.csv')
test = pd.read_csv('/kaggle/input/competitions/triagegeist/test.csv')
test_ids = test['patient_id']

In [ ]:
patient_df = pd.read_csv('/kaggle/input/competitions/triagegeist/patient_history.csv')

In [ ]:
train_merged = pd.merge(df, patient_df, on='patient_id', how='left')

test_merged = pd.merge(test, patient_df, on='patient_id', how='left')

In [ ]:
train_merged = train_merged.drop(['patient_id', 'disposition', 'ed_los_hours'], axis=1)
test_merged = test_merged.drop(['patient_id'], axis=1)

In [ ]:
X = train_merged.drop(['triage_acuity',
                       'chief_complaint_system', 'num_active_medications',
                       'systolic_bp', 'shock_index',
                       'hx_substance_use_disorder','hx_peripheral_vascular_disease'], axis=1)
y = train_merged['triage_acuity']

test = test_merged.drop([
                       'chief_complaint_system', 'num_active_medications',
                       'systolic_bp', 'shock_index',
                       'hx_substance_use_disorder','hx_peripheral_vascular_disease'], axis=1)

In [ ]:
object_cols = X.select_dtypes(include='object').columns

for col in object_cols:
    m_estimate_encoder = MEstimateEncoder(cols=[col])
    X = m_estimate_encoder.fit_transform(X, y)
    test = m_estimate_encoder.transform(test)

In [ ]:
xgb_model = XGBRegressor(random_state=42)
cat_model = CatBoostRegressor(random_state=42, verbose=0)
lgbm_model = LGBMRegressor(random_state=42)

estimators = [
    ('xgb', xgb_model),
    ('cat', cat_model),
    ('lgbm', lgbm_model)
]

meta_estimator = Ridge(random_state=42)

In [ ]:
stack_model = StackingRegressor(estimators=estimators,
                                  final_estimator=meta_estimator,
                                  cv=5)
stack_model.fit(X, y)

In [ ]:
stack_y_pred_test = stack_model.predict(test)

In [ ]:
submission_df = pd.DataFrame({'patient_id': test_ids, 'triage_acuity': stack_y_pred_test})
submission_df['triage_acuity'] = submission_df['triage_acuity'].round().astype(int)
submission_df.to_csv('submission.csv', index=False)

print("Submission file 'submission.csv' created successfully.")